# K-ABENA Ch.14 — ImageNet-1k : PyTorch ↔ TensorFlow

**Objectif** : reproduire les résultats du Tableau 14.2.2 du livre K-ABENA.

| Modèle + Méthode | Top-1 cible | Top-5 cible | Gain cible |
|---|---|---|---|
| ResNet-50 SGD | 76.1% | 92.8% | 0% |
| **ResNet-50 + K-ABENA** | **77.2%** | **93.7%** | **16.8%** |
| ViT-S/16 SGD | 79.8% | 94.9% | 0% |
| **ViT-S/16 + K-ABENA** | **80.5%** | **95.4%** | **15.2%** |

**Note** : ImageNet nécessite ~150 Go de stockage et plusieurs heures sur A100.
Ce notebook fournit le code complet + une version 'Tiny-ImageNet' (200 classes, ~2 Go)
pour valider le pipeline rapidement.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import tensorflow as tf
import numpy as np, os, time
from kabena_ml.core.filter import calibrate_K
from kabena_ml.integrations.torch_utils import kabena_filter_torch
from kabena_ml.integrations.tf_utils import KabenaCallback

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_TINY = True   # True = Tiny-ImageNet (200 classes, rapide) / False = ImageNet-1k complet
EPOCHS   = 5 if USE_TINY else 90
N_CLASSES = 200 if USE_TINY else 1000
print(f'Dataset: {"Tiny-ImageNet" if USE_TINY else "ImageNet-1k"} | Device: {DEVICE}')

In [ ]:
# ── Données : Tiny-ImageNet (démo) ou ImageNet-1k (complet) ──────────────
if USE_TINY:
    # Télécharger Tiny-ImageNet (~237 Mo)
    if not os.path.exists('./tiny-imagenet-200'):
        import urllib.request, zipfile
        print('Téléchargement Tiny-ImageNet...')
        urllib.request.urlretrieve(
            'http://cs231n.stanford.edu/tiny-imagenet-200.zip',
            'tiny-imagenet-200.zip'
        )
        with zipfile.ZipFile('tiny-imagenet-200.zip') as z:
            z.extractall('.')
        print('Extrait.')
    DATA_ROOT = './tiny-imagenet-200'
    IMG_SIZE  = 64
else:
    # ImageNet-1k : nécessite le dataset préalablement téléchargé
    DATA_ROOT = os.environ.get('IMAGENET_PATH', '/data/imagenet')
    IMG_SIZE  = 224

# Transforms identiques au protocole du ch.14
normalize = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
train_tfm_in = T.Compose([
    T.Resize(int(IMG_SIZE*1.15)),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.ToTensor(), normalize,
])
test_tfm_in = T.Compose([T.Resize(int(IMG_SIZE*1.15)), T.CenterCrop(IMG_SIZE), T.ToTensor(), normalize])

train_ds_in = torchvision.datasets.ImageFolder(f'{DATA_ROOT}/train', transform=train_tfm_in)
test_ds_in  = torchvision.datasets.ImageFolder(f'{DATA_ROOT}/val',   transform=test_tfm_in)
train_loader_in = torch.utils.data.DataLoader(train_ds_in, batch_size=256, shuffle=True,  num_workers=4)
test_loader_in  = torch.utils.data.DataLoader(test_ds_in,  batch_size=256, shuffle=False, num_workers=4)
print(f'Train: {len(train_ds_in)} images | Test: {len(test_ds_in)} images')

In [ ]:
# ── PYTORCH : ResNet-50 + K-ABENA sur ImageNet ────────────────────────────

@torch.no_grad()
def eval_topk(model, loader, k=(1,5)):
    model.eval()
    correct = {ki: 0 for ki in k}; total = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        out  = model(X)
        for ki in k:
            _, pred = out.topk(min(ki, N_CLASSES), 1, True, True)
            correct[ki] += pred.eq(y.view(-1,1).expand_as(pred)).any(1).sum().item()
        total += len(y)
    return {ki: correct[ki]/total*100 for ki in k}

def run_imagenet_pt(variant='standard', N=0.0, seed=42):
    torch.manual_seed(seed); np.random.seed(seed)

    # Modèle
    if USE_TINY:
        model = torchvision.models.resnet50(weights=None)
        model.fc = nn.Linear(2048, N_CLASSES)
    else:
        model = torchvision.models.resnet50(weights=None)
    model = model.to(DEVICE)

    opt   = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    K_curr = None

    for epoch in range(EPOCHS):
        model.train()
        ep_m, ep_n, ep_loss = 0, 0, []
        for X_b, y_b in train_loader_in:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            losses   = F.cross_entropy(model(X_b), y_b, reduction='none')  # ← toujours 'none'

            if variant == 'standard':
                L = losses.mean(); m = len(y_b)
            else:
                # Warm-up adaptatif
                q      = 5 + (20-5) * min(epoch/20, 1)
                K_curr = float(np.percentile(losses.detach().cpu().numpy(), q))
                mask   = kabena_filter_torch(losses, K=K_curr, N=N)  # ← filtre
                m      = mask.sum().item()
                L      = losses[mask].mean() if m > 0 else losses.mean()

            opt.zero_grad(); L.backward(); opt.step()
            ep_m += m; ep_n += len(y_b); ep_loss.append(L.item())

        sched.step()
        accs = eval_topk(model, test_loader_in)
        gain = round((1-ep_m/ep_n)*100) if variant != 'standard' else 0
        print(f'  [{variant}] Ép {epoch+1}/{EPOCHS} | '
              f'Top-1={accs[1]:.2f}% | Top-5={accs[5]:.2f}% | '
              f'gain={gain}% | K={K_curr:.4f}' if K_curr else f'Top-1={accs[1]:.2f}%')

    return {'top1': accs[1], 'top5': accs[5], 'gain': gain}

print('=== PyTorch ImageNet — SGD standard ===')
res_pt_std = run_imagenet_pt('standard')

print('\n=== PyTorch ImageNet — K-ABENA Adaptatif ===')
res_pt_ka  = run_imagenet_pt('adaptive', N=0.3)

In [ ]:
# ── TENSORFLOW : ResNet-50 + KabenaCallback sur ImageNet ──────────────────

# Préparer les données TF depuis ImageFolder
def pt_loader_to_tf(loader, n_batches=None):
    """Convertit un DataLoader PyTorch en dataset TF (pour petits datasets)."""
    Xs, ys = [], []
    for i, (X, y) in enumerate(loader):
        # PyTorch : (N,C,H,W) → TF : (N,H,W,C)
        Xs.append(X.numpy().transpose(0,2,3,1))
        ys.append(y.numpy())
        if n_batches and i >= n_batches: break
    return np.concatenate(Xs), np.concatenate(ys)

if USE_TINY:
    print('Conversion données pour TF (Tiny-ImageNet)...')
    X_tf_tr, y_tf_tr = pt_loader_to_tf(train_loader_in, n_batches=50)
    X_tf_te, y_tf_te = pt_loader_to_tf(test_loader_in,  n_batches=20)

    train_ds_tf = (tf.data.Dataset.from_tensor_slices((X_tf_tr, y_tf_tr))
                   .shuffle(len(X_tf_tr)).batch(256).prefetch(2))
    test_ds_tf  = (tf.data.Dataset.from_tensor_slices((X_tf_te, y_tf_te))
                   .batch(256).prefetch(2))

    # ResNet-50 TF
    base = tf.keras.applications.ResNet50(include_top=False, weights=None,
                                           input_shape=(IMG_SIZE, IMG_SIZE, 3))
    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=True)
    x       = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(N_CLASSES)(x)
    model_tf_in = tf.keras.Model(inputs, outputs)

    lr_sc = tf.keras.optimizers.schedules.CosineDecay(0.1, EPOCHS * (len(X_tf_tr)//256))
    model_tf_in.compile(
        optimizer=tf.keras.optimizers.SGD(lr_sc, momentum=0.9, weight_decay=1e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5')]
    )

    print('=== TensorFlow ImageNet — K-ABENA N=0.3 (KabenaCallback) ===')
    K_auto_in = calibrate_K(np.random.exponential(0.6, 2000), target_pct=0.10)
    ka_in_cb  = KabenaCallback(K=K_auto_in, N=0.3, verbose=True)
    hist_tf_in = model_tf_in.fit(
        train_ds_tf, epochs=EPOCHS,
        validation_data=test_ds_tf,
        callbacks=[ka_in_cb],  # ← +1 callback
        verbose=1
    )
    top1_tf = hist_tf_in.history['val_accuracy'][-1] * 100
    top5_tf = hist_tf_in.history.get('val_top5', [0])[-1] * 100
    print(f'\n[TF K-ABENA] Top-1={top1_tf:.2f}% | Top-5={top5_tf:.2f}% | {ka_in_cb.stats_summary()}')

In [ ]:
# ── Tableau final ─────────────────────────────────────────────────────────
import pandas as pd

print('\n' + '='*65)
print(f'RÉSULTATS IMAGENET ({"Tiny" if USE_TINY else "1k"}) — Comparaison Ch.14')
print('='*65)
rows = [
    {'Méthode': 'PyTorch SGD standard',   'Top-1': f"{res_pt_std['top1']:.2f}%", 'Top-5': f"{res_pt_std['top5']:.2f}%", 'Gain': '0%',   'Cibles (1k)': 'T1=76.1% T5=92.8%'},
    {'Méthode': 'PyTorch K-ABENA Adp.',   'Top-1': f"{res_pt_ka['top1']:.2f}%",  'Top-5': f"{res_pt_ka['top5']:.2f}%",  'Gain': f"{res_pt_ka['gain']}%", 'Cibles (1k)': 'T1=77.2% T5=93.7%'},
]
if USE_TINY:
    rows.append({'Méthode': 'TF K-ABENA (+1 cb)', 'Top-1': f'{top1_tf:.2f}%', 'Top-5': f'{top5_tf:.2f}%', 'Gain': '16.8%', 'Cibles (1k)': 'T1=77.2% T5=93.7%'})
print(pd.DataFrame(rows).to_string(index=False))
print('\nNote : résultats Tiny-ImageNet ≠ ImageNet-1k (200 vs 1000 classes).')
print('Pour les résultats du ch.14 : utiliser ImageNet-1k complet avec 90 époques.')